# The Gaussian Distribution

## What's covered

- The **1D Gaussian** — definition, key facts, the standard normal
- Why the Gaussian is everywhere — a sneak peek at the **Central Limit Theorem**
- The **multivariate Gaussian** — the PDF and what each piece means
- The **covariance matrix** — geometry of correlated dimensions
- **Marginal Gaussians** — the marginal of a multivariate Gaussian is Gaussian
- **Conditional Gaussians** — the conditional is also Gaussian, with closed-form mean & covariance
- **MLE for Gaussian parameters** — `\hat{\mu}` and `\hat{\Sigma}` in closed form
- Where this appears in ML — almost everywhere


## The 1D Gaussian

`X \sim \mathcal{N}(\mu, \sigma^2)` has PDF:

$$
f_X(x) = \frac{1}{\sigma \sqrt{2 \pi}} \exp\!\left(-\frac{(x - \mu)^2}{2 \sigma^2}\right)
$$

Two parameters, both interpretable:

- `\mu` = the **mean** — also the median and the mode (the Gaussian is symmetric).
- `\sigma^2` = the **variance**; `\sigma` is the standard deviation.

The **standard normal** `\mathcal{N}(0, 1)` is the special case `\mu = 0`, `\sigma = 1`. Every other Gaussian is a linear transformation of it: if `Z \sim \mathcal{N}(0, 1)`, then `X = \mu + \sigma Z \sim \mathcal{N}(\mu, \sigma^2)`. We'll use this fact constantly.

**The "68–95–99.7 rule"** (worth memorizing for interview answers):

- ~68% of mass within 1 standard deviation: `[\mu - \sigma, \mu + \sigma]`
- ~95% within 2 standard deviations
- ~99.7% within 3 standard deviations

**Why this shape?** The exponential of a negative quadratic. The quadratic makes the PDF maximum at `\mu` and quickly drives the density to zero on either side — exponentially fast in distance squared. This is what gives the Gaussian its light tails (low kurtosis, no fat outliers).

**The `\frac{1}{\sigma \sqrt{2 \pi}}` normalizer** is what makes the PDF integrate to 1. The integral `\int e^{-x^2/2} \, dx = \sqrt{2\pi}` is one of the most-quoted "non-elementary but famous" integrals — there's no closed-form antiderivative, but the definite integral has this beautiful value (provable with the polar-coordinates trick).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

# Verify the 68-95-99.7 rule
N = 500_000
mu, sigma = 5, 2
X = rng.normal(mu, sigma, N)

within_1 = np.mean(np.abs(X - mu) < 1 * sigma)
within_2 = np.mean(np.abs(X - mu) < 2 * sigma)
within_3 = np.mean(np.abs(X - mu) < 3 * sigma)
print(f"P(|X - mu| < 1*sigma) = {within_1:.4f}   (analytic 0.6827)")
print(f"P(|X - mu| < 2*sigma) = {within_2:.4f}   (analytic 0.9545)")
print(f"P(|X - mu| < 3*sigma) = {within_3:.4f}   (analytic 0.9973)")

# Standardization trick: (X - mu)/sigma ~ N(0, 1)
Z = (X - mu) / sigma
print(f"\nStandardized:  mean = {Z.mean():.4f}, std = {Z.std():.4f}")
print("Confirms X = mu + sigma * Z is the right relation.")


## Why the Gaussian is everywhere — CLT preview

The deep reason the Gaussian shows up so often: the **Central Limit Theorem**. Notebook 8 covers it formally, but the punchline now:

> The average of many independent random variables — *whatever their original distribution* — is approximately Gaussian.

This means: any quantity that's a sum or average of many small independent effects tends to be Gaussian. Measurement noise, polling errors, neural activations across many inputs, financial returns — many things in nature look Gaussian *not* because the underlying mechanism is Gaussian, but because *averaging hides the original distribution*.

A few other reasons Gaussians dominate:

- **Maximum entropy.** Among all distributions with a given mean and variance, the Gaussian has the highest entropy (the "least committed" distribution). It's the *least informative* choice for "a real-valued quantity with this much spread."
- **Closed-form arithmetic.** Affine combinations of Gaussians stay Gaussian. Marginals of Gaussians stay Gaussian. Conditionals of Gaussians stay Gaussian. Most distributions break under one of these operations; the Gaussian survives all of them. This is what makes Gaussian *processes* and Kalman filters tractable.
- **Conjugacy.** The Gaussian is conjugate to itself for the mean — making Bayesian updates trivial (notebook 6).
- **Universal approximator status (in ML).** Mixtures of Gaussians can approximate any density. Variational families default to Gaussians.

The Gaussian isn't always the right model, but it's almost always the *default* model, and you should understand exactly when and why it succeeds and fails.


## The multivariate Gaussian

The `d`-dimensional generalization. `\mathbf{X} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)` with:

- **Mean vector** `\boldsymbol{\mu} \in \mathbb{R}^d`
- **Covariance matrix** `\Sigma \in \mathbb{R}^{d \times d}` — symmetric, positive definite

PDF:

$$
f_{\mathbf{X}}(\mathbf{x}) = \frac{1}{(2 \pi)^{d/2} |\Sigma|^{1/2}} \exp\!\left(-\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu})\right)
$$

Reading the pieces:

- `(\mathbf{x} - \boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu})` is the **Mahalanobis distance squared** between `\mathbf{x}` and `\boldsymbol{\mu}`. Unlike Euclidean distance, this distance accounts for correlation — it stretches and rotates space according to `\Sigma`. Equal-density contours are *ellipsoids* whose axes align with the eigenvectors of `\Sigma`.
- `|\Sigma|^{1/2}` (square root of the determinant) is the volume scaling — bigger eigenvalues, bigger ellipsoid, smaller peak density.
- `(2 \pi)^{d/2}` ensures the integral is 1.

**Special cases.**

- **Spherical covariance** `\Sigma = \sigma^2 I` — equal variance in every direction, no correlations. Contours are circles/spheres.
- **Diagonal covariance** `\Sigma = \text{diag}(\sigma_1^2, \dots, \sigma_d^2)` — different variances per dimension, but no cross-correlation. Contours are axis-aligned ellipses.
- **General covariance** — any symmetric PSD `\Sigma`. Contours are rotated, stretched ellipses.

This last case is why understanding covariance matrices = understanding the *geometry* of multivariate Gaussians.


In [ ]:
# Three 2D Gaussian shapes
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
mean = np.array([0.0, 0.0])

shapes = [
    ("Spherical: Σ = I",         np.eye(2)),
    ("Diagonal: Σ = diag(1, 4)", np.diag([1.0, 4.0])),
    ("Correlated: ρ = 0.8",      np.array([[1.0, 0.8], [0.8, 1.0]])),
]
for ax, (name, cov) in zip(axes, shapes):
    samples = rng.multivariate_normal(mean, cov, 5000)
    ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3)
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
    ax.set_aspect("equal")
    ax.set_title(name)
plt.tight_layout(); plt.show()

print("Notice how the eigenstructure of Σ shapes the cloud:")
print("- spherical → circular cloud")
print("- diagonal → axis-aligned ellipse")
print("- correlated → ellipse tilted along the dominant eigenvector")


## The covariance matrix — geometry decoded

For `\mathbf{X} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)`:

- `\Sigma_{ii} = \text{Var}(X_i)` — variance along axis `i`.
- `\Sigma_{ij} = \text{Cov}(X_i, X_j)` — covariance between dimensions `i` and `j`.
- `\Sigma` is symmetric and positive definite (always — otherwise no density exists).

**Eigendecomposition.** Write `\Sigma = Q \Lambda Q^\top` (spectral theorem from the linear algebra repo). Then:

- The **eigenvectors** (columns of `Q`) are the **principal axes** of the equal-density ellipses.
- The **eigenvalues** (diagonal of `\Lambda`) are the **variances along those axes**.
- The lengths of the ellipsoid semi-axes are `\sqrt{\lambda_i}` (or some constant multiple, depending on which density level you're picturing).

So **PCA on a Gaussian cloud finds exactly the axes of the equal-density ellipsoids**. This is not coincidence — PCA *is* the spectral decomposition of the sample covariance matrix.

**A handy property — affine transformations.** If `\mathbf{X} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)`, then for any matrix `A` and vector `\mathbf{b}`,

$$
A \mathbf{X} + \mathbf{b} \sim \mathcal{N}(A \boldsymbol{\mu} + \mathbf{b}, \, A \Sigma A^\top)
$$

Gaussian → linear transformation → still Gaussian. This is why Gaussian processes work, why Kalman filters are tractable, and why Variational Autoencoders use Gaussian latents — the math closes nicely.


In [ ]:
# PCA on the correlated Gaussian recovers its eigenstructure
cov = np.array([[1.0, 0.8],
                [0.8, 1.0]])
samples = rng.multivariate_normal([0, 0], cov, 50_000)

# Sample covariance
S = np.cov(samples.T)
print(f"True Σ:\n{cov}")
print(f"Sample Σ:\n{S.round(4)}")

# Eigendecomposition
eigvals, eigvecs = np.linalg.eigh(S)
print(f"\nEigenvalues (variances along principal axes): {eigvals.round(4)}")
print(f"Eigenvectors (principal axes):\n{eigvecs.round(4)}")
print("\nLargest eigenvalue ≈ 1.8 → ellipse stretched along (1, 1)/√2")
print("Smallest eigenvalue ≈ 0.2 → ellipse compressed along (-1, 1)/√2")


## Marginal Gaussians are Gaussian

Partition the random vector as `\mathbf{X} = (\mathbf{X}_a, \mathbf{X}_b)` and the parameters in conformable blocks:

$$
\boldsymbol{\mu} = \begin{bmatrix} \boldsymbol{\mu}_a \\ \boldsymbol{\mu}_b \end{bmatrix}, \qquad
\Sigma = \begin{bmatrix} \Sigma_{aa} & \Sigma_{ab} \\ \Sigma_{ba} & \Sigma_{bb} \end{bmatrix}
$$

Then the **marginal distribution of `\mathbf{X}_a`** is just

$$
\mathbf{X}_a \sim \mathcal{N}(\boldsymbol{\mu}_a, \, \Sigma_{aa})
$$

That's it. Drop the rows and columns for `\mathbf{X}_b`; the marginal of a Gaussian is the corresponding sub-Gaussian. No integration required.

Compare this to general distributions, where marginalization is an integral that may be intractable. The Gaussian gives it to you for free.


## Conditional Gaussians — the closed form

The single most-used result about multivariate Gaussians. With the same block partition, the conditional of `\mathbf{X}_a` given `\mathbf{X}_b = \mathbf{x}_b` is

$$
\mathbf{X}_a | (\mathbf{X}_b = \mathbf{x}_b) \sim \mathcal{N}\bigl(\boldsymbol{\mu}_{a | b}, \, \Sigma_{a | b}\bigr)
$$

with

$$
\boldsymbol{\mu}_{a | b} = \boldsymbol{\mu}_a + \Sigma_{ab} \Sigma_{bb}^{-1} (\mathbf{x}_b - \boldsymbol{\mu}_b)
$$

$$
\Sigma_{a | b} = \Sigma_{aa} - \Sigma_{ab} \Sigma_{bb}^{-1} \Sigma_{ba}
$$

Two observations worth pausing on:

- The **conditional mean** is the unconditional mean `\boldsymbol{\mu}_a` *plus* a correction that's linear in the observed value `\mathbf{x}_b`. The slope of that correction is `\Sigma_{ab} \Sigma_{bb}^{-1}` — exactly the **regression coefficients** of `\mathbf{X}_a` on `\mathbf{X}_b`. In other words, **the conditional mean of a joint Gaussian is the best linear regressor**.
- The **conditional covariance** doesn't depend on `\mathbf{x}_b` at all — only on the structure of `\Sigma`. The uncertainty is the same no matter what we observed. This is a Gaussian-specific property and the reason it's so well-behaved.

**Schur complement.** `\Sigma_{aa} - \Sigma_{ab} \Sigma_{bb}^{-1} \Sigma_{ba}` is the **Schur complement** of `\Sigma_{bb}` in `\Sigma`. It comes up all over the place in optimization and statistics; this is one of its first big appearances.

**ML angle.** This formula is the math behind:

- **Linear regression closed form.** Treat `(X, Y)` as joint Gaussian; the conditional mean `\mathbb{E}[Y | X]` is the linear regression function.
- **Gaussian processes (GPs).** Train data and test data are jointly Gaussian; predictions are conditional means; uncertainty estimates are conditional covariances.
- **Kalman filters.** Each update is a Gaussian-conditional-on-observation step using exactly these formulas.
- **Conditional VAEs.** Some variants use Gaussian-conditional structures.


In [ ]:
# Empirically verify the Gaussian conditional formula
mu = np.array([1.0, 2.0])
Sigma = np.array([[1.0, 0.7],
                  [0.7, 2.0]])

N = 500_000
joint_samples = rng.multivariate_normal(mu, Sigma, N)

# Condition on X2 ≈ 3.0
x2_obs = 3.0
band = np.abs(joint_samples[:, 1] - x2_obs) < 0.05
X1_cond = joint_samples[band, 0]

# Analytical conditional N(mu_1|2, sigma_1|2)
mu_1, mu_2 = mu
S11, S12 = Sigma[0, 0], Sigma[0, 1]
S22 = Sigma[1, 1]
mu_cond = mu_1 + S12 / S22 * (x2_obs - mu_2)
sigma2_cond = S11 - S12 / S22 * S12
print(f"Analytical: X1 | X2 = {x2_obs} ~ N({mu_cond:.4f}, {sigma2_cond:.4f})")
print(f"Empirical : mean = {X1_cond.mean():.4f},  var = {X1_cond.var(ddof=1):.4f}")
print(f"           ({len(X1_cond)} samples used)")


## MLE for Gaussian parameters

Given i.i.d. samples `\mathbf{x}_1, \dots, \mathbf{x}_N` from `\mathcal{N}(\boldsymbol{\mu}, \Sigma)`, the maximum-likelihood estimators are clean and intuitive:

$$
\hat{\boldsymbol{\mu}}_{\text{MLE}} = \bar{\mathbf{x}} = \frac{1}{N} \sum_{i=1}^N \mathbf{x}_i
$$

$$
\hat{\Sigma}_{\text{MLE}} = \frac{1}{N} \sum_{i=1}^N (\mathbf{x}_i - \bar{\mathbf{x}})(\mathbf{x}_i - \bar{\mathbf{x}})^\top
$$

The sample mean estimates the population mean. The sample covariance estimates the population covariance — outer-product form, which makes it clearly a sum of rank-1 matrices.

**The `N` vs `N - 1` debate.** As in the 1D case, MLE uses `N` in the denominator but it's biased downward. The **unbiased** estimator uses `N - 1` (Bessel's correction). For large `N` the difference is negligible; for small `N` and in formal statistics, you'll see `N - 1`. NumPy's `np.cov` uses `N - 1` by default (`ddof=1`).

**Derivation sketch.** Log-likelihood:

$$
\log p(\{\mathbf{x}_i\} | \boldsymbol{\mu}, \Sigma) = -\frac{Nd}{2} \log(2\pi) - \frac{N}{2} \log |\Sigma| - \frac{1}{2} \sum_i (\mathbf{x}_i - \boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x}_i - \boldsymbol{\mu})
$$

Set `\partial / \partial \boldsymbol{\mu} = 0` → `\hat{\boldsymbol{\mu}} = \bar{\mathbf{x}}`. Set `\partial / \partial \Sigma^{-1} = 0` (easier than `\partial / \partial \Sigma`) → the closed form above. The matrix calculus identities from the calculus repo, notebook 5, are exactly what you need.

**Note.** The MLE for `\Sigma` requires `N \geq d` to be invertible. With more dimensions than samples (`N < d`), the sample covariance is singular and you need to regularize (e.g., add `\lambda I`, which corresponds to a Gaussian prior on `\Sigma`). This is the *shrinkage* idea behind Ledoit-Wolf and Tikhonov regularization.


In [ ]:
# Verify MLE for multivariate Gaussian
N, d = 1000, 3
true_mu = np.array([1.0, -2.0, 0.5])
true_cov = np.array([[2.0, 0.5, 0.0],
                     [0.5, 1.5, 0.3],
                     [0.0, 0.3, 1.0]])

samples = rng.multivariate_normal(true_mu, true_cov, N)
mu_hat = samples.mean(axis=0)
cov_hat = np.cov(samples.T, ddof=1)

print("MLE for mean:")
print(f"  true: {true_mu}")
print(f"  est:  {mu_hat.round(4)}")
print(f"\nMLE for covariance:")
print(f"  true:\n{true_cov}")
print(f"  est (unbiased):\n{cov_hat.round(4)}")

# Max diff between true and estimated
print(f"\nmax |true - est| in covariance: {np.max(np.abs(true_cov - cov_hat)):.4f}")


## Where this appears in ML

Almost everywhere. Of all the distributions in the previous notebook, the Gaussian gets its own notebook because it's the *most* used.

- **Mean squared error regression.** Equivalent to MLE under `Y | X \sim \mathcal{N}(\mathbf{w}^\top \mathbf{x}, \sigma^2)`. *Why MSE is the default loss.*
- **Linear discriminant analysis (LDA).** Class-conditional `p(x | y)` is Gaussian per class with shared covariance. Bayes' rule recovers a linear classifier.
- **Quadratic discriminant analysis (QDA).** Same as LDA but each class has its own covariance.
- **Gaussian mixture models.** `p(x) = \sum_k \pi_k \mathcal{N}(\mathbf{x} | \boldsymbol{\mu}_k, \Sigma_k)`. The Gaussian's flexibility comes from mixing.
- **PCA.** Maximum-likelihood projection under a Gaussian model with low-rank covariance.
- **Gaussian processes.** The output at every input is jointly Gaussian; predictions use the conditional formula directly.
- **Kalman filters.** State estimates are Gaussian; updates are Gaussian conditionals on observations. The Kalman gain *is* `\Sigma_{ab} \Sigma_{bb}^{-1}`.
- **VAEs.** Encoder outputs `q(z | x) = \mathcal{N}(\boldsymbol{\mu}_\phi(x), \Sigma_\phi(x))`; the prior is `\mathcal{N}(0, I)`; the KL term has closed form because both are Gaussian.
- **Diffusion models.** Forward and (approximated) reverse processes are Gaussian. Training objective is a sum of MSE-like terms (Gaussian log-likelihoods).
- **Laplace approximation.** Approximate any posterior by a Gaussian at its mode using the Hessian of the log-density (notebook 6 / calculus notebook 6).
- **Bayesian linear regression.** Gaussian prior + Gaussian likelihood → Gaussian posterior. Closed form, no sampling needed.
- **Weight initialization.** Xavier and Kaiming initializations sample weights from carefully tuned Gaussians (variance-preserving across layers).
- **Gaussian noise injection.** Dropout, label smoothing, data augmentation often use Gaussian noise for analytical tractability.
- **Normalizing flows.** Base distribution is almost always standard Gaussian; flows transform it into something arbitrary while keeping likelihoods tractable.
- **Reparameterization trick.** Sampling `z \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)` becomes `z = \boldsymbol{\mu} + L \boldsymbol{\epsilon}` for `\boldsymbol{\epsilon} \sim \mathcal{N}(0, I)` — gives differentiable samples for VAEs.

Next notebook: **maximum likelihood and Bayesian inference** — we formalize the MLE we just glimpsed, contrast it with Bayesian inference, and connect both back to the loss functions you train with every day.
